# Module 8 Assignment: Give Your Persona Agent Memory With Sessions

Lab 7 added a `SQLiteSession` to Study Buddy: first in memory, to fix two `Runner.run` calls that
did not remember each other, then file backed, so the conversation survives a kernel restart. The
Gradio `chat()` function got simpler in the process, no more rebuilding `history` by hand.

This assignment makes you do the same thing to your **own** Persona Agent, the one you rebuilt on
the OpenAI Agents SDK in the Module 7 assignment. Nothing about your agent's persona, notes, or
three tools needs to change here: you are giving it memory, not redesigning it.

**What you'll build:**
1. Your Module 7 assignment's Agent, Azure configuration, and three tools, carried over unchanged
2. A quick demonstration that two separate `Runner.run` calls do not remember each other
3. A file backed `SQLiteSession` that fixes it
4. A simplified Gradio `chat()` function that uses `session=session` instead of rebuilding `history`

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu`, same as every lab and assignment so
far.

If you get stuck on any step, look at how Lab 7 made the same kind of change to Study Buddy, and do
the analogous thing to your own agent.

## 0. Install the packages (run once)

Same as Lab 7: no new package beyond what your Module 7 assignment already installed.
`SQLiteSession` ships with `openai-agents` itself.

In [ ]:
!uv pip install --upgrade openai openai-agents pypdf python-dotenv requests gradio

## 1. Imports

**TODO:** Copy the imports from your Module 7 assignment, plus one new name: `SQLiteSession`,
imported from `agents` alongside `Agent` and `Runner`, the same way Lab 7, Step 1 added it.

In [1]:
from dotenv import load_dotenv
import os
import asyncio
import html
from typing import Literal, Optional

import requests
from openai import AsyncAzureOpenAI
from agents import (
    Agent,
    Runner,
    function_tool,
    OpenAIChatCompletionsModel,
    set_tracing_disabled,
    SQLiteSession,
)
from pypdf import PdfReader
import gradio as gr


c:\Users\yoyou\OneDrive\Documents\Cursor Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load your Azure OpenAI credentials from `.env`

Identical to the Module 7 assignment. Sessions do not change which provider you talk to.

In [2]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
DEPLOYMENT = "gpt-5.4-ptu"

print("Endpoint loaded:", bool(AZURE_OPENAI_ENDPOINT))
print("API key loaded:", bool(AZURE_OPENAI_API_KEY))

Endpoint loaded: True
API key loaded: True


## 3. Configure the OpenAI Agents SDK for Azure OpenAI

**TODO:** Copy this setup from your Module 7 assignment, unchanged: `set_tracing_disabled(True)`,
your `AsyncAzureOpenAI` client, and `OpenAIChatCompletionsModel`. A session is a feature of
`Runner.run`, not of the model or the client, so nothing here needs to change.

In [3]:
set_tracing_disabled(True)

azure_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version="2024-10-21",
)

azure_model = OpenAIChatCompletionsModel(
    model=DEPLOYMENT,
    openai_client=azure_client,
)

print("Azure model configured for the OpenAI Agents SDK.")

Azure model configured for the OpenAI Agents SDK.


## 4. Your persona, notes, and three tools

**TODO:** Bring over your persona, your notes, and all three of your `@function_tool` decorated
tools from the Module 7 assignment, unchanged. None of them need to change for this assignment: a
session only affects how conversation history is stored and replayed, not what a tool does.

In [ ]:
persona = (
    "Mod5 Bot, a rude assistant that drills you in order to make sure you understand the material."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
STUDY_FILE = "study_this.txt"
NOTES_PATH = "notes/notes.pdf"
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:300], "...")

In [ ]:
@function_tool
def my_tool(question: str) -> str:
    """Record a question the student got wrong or didn't know to study_this.txt.

    Args:
        question: The exact question the student missed.
    """
    print(f"Tool called to log when you get an answer wrong: {question}")
    with open(STUDY_FILE, "a", encoding="utf-8") as f:
        f.write(question + "\n")
    return "Wrong answer recorded"

print("Tool ready:", my_tool.name)

In [ ]:
OPENTDB_CATEGORIES = {"general_knowledge": 9}

@function_tool
def api_tool(category: Optional[Literal["general_knowledge"]] = None) -> str:
    """Fetch a trivia question from an external trivia database when the student asks for a break.

    Args:
        category: Trivia category to pull from (general_knowledge).
    """
    print(f"Tool called to fetch a trivia question (category={category})")
    params = {"amount": 1, "type": "multiple"}
    if category in OPENTDB_CATEGORIES:
        params["category"] = OPENTDB_CATEGORIES[category]

    try:
        response = requests.get("https://opentdb.com/api.php", params=params, timeout=20)
        data = response.json()
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"api_tool network error: {e}")
        return "The trivia API is temporarily unavailable, get back to studying your notes."

    if data.get("response_code") != 0 or not data.get("results"):
        return "Could not fetch a trivia question right now, stop slacking and get back to the quiz."

    q = data["results"][0]
    question = html.unescape(q["question"])
    answer = html.unescape(q["correct_answer"])
    return f"Trivia question: {question} (correct answer: {answer})"

print("Tool ready:", api_tool.name)

In [ ]:
@function_tool
def third_tool() -> str:
    """Check how many questions have been logged as missed in study_this.txt."""
    print("Tool called to check review progress")
    try:
        with open(STUDY_FILE, "r", encoding="utf-8") as f:
            missed = [line for line in f if line.strip()]
    except FileNotFoundError:
        missed = []

    if not missed:
        return "No questions logged yet, surprisingly."
    return f"You have logged {len(missed)} missed question(s) in study_this.txt."

print("Tool ready:", third_tool.name)

## 5. Build your Agent

**TODO:** Copy this from your Module 7 assignment, unchanged: `name`, `instructions`, your `tools`
list, and `model=azure_model`.

In [ ]:
instructions = f"""
# Your role

You are {persona}

You ask questions on the provided notes to test knowledge.

# Context

Here are the notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong, staying in character the whole time.

If the student's answer was wrong, or they say "I don't know", you must record it as a missed question using my_tool before moving on to the next question.

If the student asks for a break from the notes, use api_tool to fetch a trivia question, this does not need to be related to the notes.

If the student asks how they are doing, use third_tool to see how many questions they have logged so far.

IMPORTANT: only ask quiz questions from the notes above. If the student asks about something outside the notes that doesn't involve any other tools, tell them it's not related to the subject at hand.
"""

mod5_agent = Agent(
    name="Mod5 Bot",
    instructions=instructions,
    tools=[my_tool, api_tool, third_tool],
    model=azure_model,
)

print("Agent built:", mod5_agent.name)

## 6. Confirm the problem: two calls that do not know each other

**TODO:** Run your agent twice in a row with `await Runner.run(your_agent, "...")`, the same way
Lab 7, Step 7 did with Study Buddy. Pick a first message that gives the agent some information (for
example, ask it a question whose answer it needs to remember), then a second message that only
makes sense if the agent remembers the first (for example, "what did I just ask you?" or "say that
again"). Confirm the second reply shows the agent has no idea what you mean.

In [ ]:
result = await Runner.run(mod5_agent, "Quiz me.")
print(result.final_output)

In [ ]:
result = await Runner.run(mod5_agent, "I dunno.")
print(result.final_output)


## 7. Fix it with a file backed `SQLiteSession`

**TODO:** Create a `SQLiteSession`, the same way Lab 7, Step 9 did: give it a `session_id` string,
and a `db_path` so the conversation is written to a file and survives a kernel restart. Then repeat
the same two messages from Step 6, this time passing `session=your_session` to both calls, and
confirm the second reply now correctly remembers the first.

In [ ]:
mod5_session = SQLiteSession("mod5_bot_quiz", "conversation_history.db")

In [ ]:
result = await Runner.run(
    mod5_agent,
    "Quiz me",
    session=mod5_session,
)
print(result.final_output)

In [ ]:
result = await Runner.run(
    mod5_agent,
    "I dunno.",
    session=mod5_session,
)
print(result.final_output)

## 8. Give it a real chat UI with Gradio

**TODO:** Copy this skeleton from Lab 7, Step 11 exactly, structure and all. Notice what is
different from your Module 7 assignment's version: `history` is no longer rebuilt inside `chat()`
at all, `session=your_session` already gives `Runner.run` everything it needs to remember the
conversation.

Try a conversation that would trigger more than one of your three tools across a few turns, then
interrupt the cell (stop button) when you are done chatting.

In [ ]:
async def chat(message, history):
    result = await Runner.run(mod5_agent, message, session=mod5_session)
    return result.final_output

gr.ChatInterface(chat).launch(inbrowser=True)

## Reflection Questions

Answer these in a new Markdown cell at the end of your notebook:

1. In Step 6, what exactly did your agent get wrong on the second call, and why? Point to the
   specific piece of information it had no way of knowing.
2. Compare your Gradio `chat()` function here to the one in your Module 7 assignment. What code did
   you get to delete, and what does `session=your_session` do in its place?
3. Your `your_session` was created with a `db_path`. If you had left `db_path` out entirely, what
   would still work the same, and what would break the next time you restarted the kernel?
4. `SQLiteSession` also has a `pop_item()` method, demonstrated in Lab 7, Step 10, for undoing the
   last turn. Describe one realistic situation in your own agent's domain where you would want to
   call it.

## Submission Checklist

- Remove your `.env` file before submitting; it contains your API key.
- Remove any `.db` file your session created; it may contain real conversation content.
- Run the notebook top to bottom and confirm it executes cleanly from a fresh kernel.
- Confirm Step 6 shows the agent failing to remember, and Step 7 shows the same scenario working
  correctly once `session=your_session` is added.
- Answer all four reflection questions in a Markdown cell at the end of the notebook.
- Rename the file to include your name, for example `Module8Assignment_JaneDoe.ipynb`, then submit
  it.

Runner.run is stateless by default, so there was no context saved from step 1 for the agent to go on.

In Module 7, chat had to manually reconstruct the conversation history list on every message. Using a SQLiteSession lets Runner.run automatically retrieve past context and store new turns.

Without db_path, the session would only exist while the venv session is active, so it would disappear when restarting the kernel. Providing conversation_history.db permanently stores it, which would allow the session to persist after restartign the venv.

If a user accidentally sends a prompt, calling session.pop_item() twice removes both the prompt and output, keeping the context clean